# **Programa Especializado en Credit Scoring con Python**
<img src="../../figuras/logo.png" width="200"/>

## 📊 **Sesión 15: Introducción a la Metodología IFRS 9 y Calibración de PD**

**Docente**: Enzo Infantes Zúñiga  
**Contacto**: <enzo.infantes28@gmail.com>  
**LinkedIn**: [enzo-infantes](https://www.linkedin.com/in/enzo-infantes/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.special import ndtri, ndtr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2E86AB', '#E84855', '#3BB273', '#F18F01', '#7B2D8B']
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

absolute_path = os.path.dirname(os.path.dirname(os.getcwd()))
data_path = os.path.join(absolute_path, "data", "s15")

np.random.seed(42)

## 🎯 Objetivos de la Sesión

Al finalizar esta sesión, el estudiante será capaz de:

1. Comprender qué es IFRS 9
2. Entender el modelo de **Pérdida Crediticia Esperada (ECL)** y sus tres etapas
3. Diferenciar la PD **Point-in-Time (PIT)** de la PD **Through-the-Cycle (TTC)**
4. Aplicar técnicas de calibración de PD bajo el marco de IFRS 9
5. Calcular la PD a 12 meses y la PD lifetime con datos reales

## **1. ¿Qué es IFRS 9 y por qué existe?**


### **1.1 El problema que IFRS 9 vino a resolver**

Para entender IFRS 9, hay que entender primero su predecesor: **IAS 39**. Bajo ese estándar anterior, las instituciones financieras solo reconocían las pérdidas crediticias cuando había **evidencia objetiva de deterioro**, es decir, cuando el incumplimiento ya había ocurrido. Esto se conoce como el enfoque de **pérdida incurrida**.

El problema fue evidente durante la crisis financiera de 2007–2008: los bancos reconocieron las pérdidas demasiado tarde

**IFRS 9** (International Financial Reporting Standard 9), vigente desde el 1 de enero de 2018, introdujo un cambio de paradigma: en lugar de esperar a que las pérdidas ocurran, ahora hay que **anticiparlas y provisionarlas desde el primer día**.


### **1.2 El núcleo de IFRS 9: El modelo ECL (Expected Credit Loss)**

El corazón de IFRS 9 es el **modelo de Pérdida Crediticia Esperada (ECL)**, que exige que las instituciones financieras reconozcan las pérdidas esperadas a lo largo de la vida del instrumento, y no solo cuando ya ocurrieron.

La fórmula general del ECL es:

$$ECL = PD \times LGD \times EAD$$

Donde:
- **PD** (Probability of Default): Probabilidad de que el deudor incumpla
- **LGD** (Loss Given Default): Porcentaje de la exposición que se pierde si hay incumplimiento
- **EAD** (Exposure at Default): Monto total expuesto en el momento del incumplimiento

### **1.3 Las Tres Etapas (Stages) de IFRS 9**

IFRS 9 clasifica los instrumentos financieros en tres etapas según el nivel de deterioro crediticio:

| Etapa | Descripción | Reconocimiento ECL |
|:---:|:---|:---:|
| **Stage 1** | Sin deterioro significativo desde el reconocimiento inicial | ECL a **12 meses** |
| **Stage 2** | Incremento significativo en el riesgo de crédito | ECL **Lifetime** |
| **Stage 3** | Activo deteriorado (default) | ECL **Lifetime** |

El concepto clave aquí es que la mayoría de las carteras comienzan en Stage 1 al originarse. Si el riesgo del cliente **aumenta significativamente** respecto al momento de la originación, la operación migra a Stage 2, y si entra en default, pasa a Stage 3.

**Implicación práctica:** Necesitamos dos versiones de la PD:
- Una PD a **12 meses** (para Stage 1)
- Una PD **lifetime** o de vida completa (para Stages 2 y 3)

## **2. PD Bajo IFRS 9: PIT vs TTC**

Este es uno de los conceptos más importantes de la metodología IFRS 9. Existen dos grandes enfoques para estimar la PD:

### **2.1 PD Through-the-Cycle (TTC)**

La PD **TTC** representa la probabilidad de default promediada a lo largo de un ciclo económico completo (expansión + recesión). Es la PD que típicamente producen los modelos de Basilea II/III para capital regulatorio.

- **Ventaja:** Es estable en el tiempo, no salta con los ciclos económicos
- **Desventaja:** No refleja el estado actual de la economía, lo cual es problemático para el reconocimiento contable de IFRS 9

### **2.2 PD Point-in-Time (PIT)**

La PD **PIT** refleja la probabilidad de default en un momento específico del tiempo, incorporando las condiciones macroeconómicas actuales y esperadas.

- **Ventaja:** Captura la realidad económica del momento, que es lo que exige IFRS 9
- **Desventaja:** Es más volátil y difícil de modelar

**IFRS 9 exige PD PIT** porque el estándar requiere que las estimaciones incorporen información hacia adelante (*forward-looking information*), incluyendo condiciones macroeconómicas actuales y proyectadas.

### **2.3 ¿Cúal es la PD final?**

La PD final es una mezcla de ambos conceptos. Además, debe incorpora ajuste macroeconómicos. Si tenemos la PD TTC, PD PIT y PD macro (PD en base a las proyecciones macroeconónicas), se deriva la PD bajo IFRS 9: 

$$\text{PD}_{final} = \text{PD}_{TTC} \times \frac{\text{PD}_{macro}}{\text{PD}_{PIT}}$$

## **3. El Proceso de Calibración de PD Bajo IFRS 9**

La **calibración** consiste en ajustar las probabilidades que produce el modelo de scoring para que sean coherentes con las tasas de default reales observadas y con las condiciones macroeconómicas esperadas. Este proceso tiene varias etapas:

1. **Obtener la PD del modelo de scoring** (probabilidades de default crudas, de naturaleza TTC)
2. **Calibración TTC:** ajustar la PD cruda a la tasa de default histórica observada (factor de escala)
3. **PD PIT:** escalar la PD TTC a las condiciones económicas *actuales* del período de reporte
4. **PD macro:** escalar la PD TTC a las condiciones económicas *proyectadas* (información forward-looking)
5. **PD final IFRS 9:** combinar los tres elementos con la fórmula $\text{PD}_{final} = \text{PD}_{TTC} \times \dfrac{\text{PD}_{macro}}{\text{PD}_{PIT}}$
6. **PD Lifetime:** proyectar la PD final a través del horizonte de vida del crédito (Stage 2/3)

Ahora implementemos cada uno de estos pasos con datos simulados que representan una cartera de créditos de consumo.

### **3.1  Dataset de Cartera de Créditos**
Simulamos una cartera de **15,000 créditos de consumo**. Cada crédito tiene:
- Variables de perfil del cliente (edad, ingreso, score de buró, etc.)
- Un indicador de default observado al cierre de cada año
- La probabilidad asignada por el modelo de scoring (PD del modelo)

In [ ]:
df = pd.read_csv(os.path.join(data_path, "credit_data_raw.csv"))

In [ ]:
print(f"Tamaño de la cartera : {len(df):,} créditos")
print(f"Tasa de default real : {df['default_obs'].mean():.2%}")
print(f"PD promedio (modelo) : {df['pd_modelo'].mean():.2%}")
print()
df.head(8)

### **3.2 Segmentación por Score Buckets**

El primer paso de la calibración es segmentar la cartera en grupos (deciles o tramos de score) y comparar la **PD estimada por el modelo** con la **tasa de default observada** en cada grupo. Si el modelo está bien calibrado, ambas deberían ser similares. Si no, necesitamos aplicar un factor corrector.

In [ ]:
# ============================================================
# PASO 1: SEGMENTACIÓN EN DECILES DE PD Y ANÁLISIS DE CALIBRACIÓN
# ============================================================

# Creamos 10 grupos (deciles) basados en la PD del modelo
df['decil_pd'] = pd.qcut(df['pd_modelo'], q=10, labels=[f'D{i}' for i in range(1, 11)])

# Calculamos, por decil: PD promedio del modelo y tasa de default observada
calibracion = df.groupby('decil_pd', observed=True).agg(
    n_creditos    = ('id_credito', 'count'),
    pd_modelo_avg = ('pd_modelo', 'mean'),
    default_rate  = ('default_obs', 'mean')
).reset_index()

# El ratio entre la tasa observada y la PD del modelo nos indica el sesgo
calibracion['ratio_calibracion'] = calibracion['default_rate'] / calibracion['pd_modelo_avg']

print("=" * 65)
print("ANÁLISIS DE CALIBRACIÓN POR DECIL")
print("=" * 65)
print(calibracion.to_string(index=False, float_format='{:.4f}'.format))
print("=" * 65)

# Ratio global de calibración
ratio_global = df['default_obs'].mean() / df['pd_modelo'].mean()
print(f"\nRatio de calibración global : {ratio_global:.4f}")
print(f"La PD del modelo está {'sobreestimando' if ratio_global < 1 else 'subestimando'} el riesgo real")

In [ ]:
# ============================================================
# VISUALIZACIÓN: PD modelo vs Tasa observada por decil
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Gráfico 1: Comparación PD modelo vs Default real ---
ax = axes[0]
x = np.arange(len(calibracion))
width = 0.38
bars1 = ax.bar(x - width/2, calibracion['pd_modelo_avg'], width,
               label='PD Modelo (TTC)', color=PALETTE[0], alpha=0.85)
bars2 = ax.bar(x + width/2, calibracion['default_rate'], width,
               label='Tasa Default Observada', color=PALETTE[1], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(calibracion['decil_pd'], fontsize=9)
ax.set_xlabel('Decil de PD')
ax.set_ylabel('Probabilidad de Default')
ax.set_title('Calibración: PD Modelo vs Tasa Default Real')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
ax.legend()
ax.grid(False)

# --- Gráfico 2: Ratio de calibración por decil ---
ax2 = axes[1]
colores = [PALETTE[2] if r >= 0.9 and r <= 1.1 else PALETTE[1] for r in calibracion['ratio_calibracion']]
ax2.bar(x, calibracion['ratio_calibracion'], color=colores, alpha=0.85, edgecolor='white')
ax2.axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='Calibración perfecta (ratio = 1)')
ax2.axhline(0.9, color='grey', linestyle=':', linewidth=1.0)
ax2.axhline(1.1, color='grey', linestyle=':', linewidth=1.0)
ax2.fill_between([-0.5, 9.5], 0.9, 1.1, alpha=0.08, color='green', label='Banda tolerancia ±10%')
ax2.set_xticks(x)
ax2.set_xticklabels(calibracion['decil_pd'], fontsize=9)
ax2.set_xlabel('Decil de PD')
ax2.set_ylabel('Ratio Tasa Observada / PD Modelo')
ax2.set_title('Ratio de Calibración por Decil')
ax2.legend(fontsize=9)
ax2.grid(False)

# Parche de leyenda para colores
patch_ok = mpatches.Patch(color=PALETTE[2], alpha=0.85, label='Bien calibrado')
patch_no = mpatches.Patch(color=PALETTE[1], alpha=0.85, label='Fuera de banda')
ax2.legend(handles=[patch_ok, patch_no] + ax2.get_legend_handles_labels()[0][-2:], fontsize=9)

plt.suptitle('Análisis de Calibración de PD — Cartera de Consumo', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### **3.3 Calibración de la PD TTC: Factor de Escala**

El método más directo de calibración es multiplicar la PD del modelo por un **factor de escala (scaler)** calculado como el ratio entre la tasa de default observada y la PD promedio del modelo. Esto se puede hacer de forma **global** (un solo factor para toda la cartera) o por **segmentos** (un factor por grupo de riesgo).

In [ ]:
# ============================================================
# PASO 2: CALIBRACIÓN TTC — FACTOR DE ESCALA
# ============================================================

# Opción A: Factor de escala global
scaler_global = df['default_obs'].mean() / df['pd_modelo'].mean() 
df['pd_ttc_calibrada'] = (df['pd_modelo'] * scaler_global).clip(0.001, 0.999)

print("CALIBRACIÓN TTC — RESULTADOS")
print("=" * 45)
print(f"  PD modelo (antes de calibrar) : {df['pd_modelo'].mean():.4%}")
print(f"  Tasa de default observada     : {df['default_obs'].mean():.4%}")
print(f"  Factor de escala global       : {scaler_global:.4f}")
print(f"  PD TTC calibrada (promedio)   : {df['pd_ttc_calibrada'].mean():.4%}")
print("=" * 45)

# Verificamos que la calibración mejoró el ajuste
calibracion['pd_ttc_calibrada'] = calibracion['pd_modelo_avg'] * scaler_global
calibracion['ratio_post_calib'] = calibracion['default_rate'] / calibracion['pd_ttc_calibrada']
print("\nVerificación post-calibración por decil:")
print(calibracion[['decil_pd', 'default_rate', 'pd_ttc_calibrada', 'ratio_post_calib']].to_string(index=False, float_format='{:.4f}'.format))

### **3.4 PD PIT, PD Macro y la Fórmula IFRS 9**

Hasta aquí tenemos una **PD TTC bien calibrada** históricamente. Sin embargo, IFRS 9 exige que las estimaciones incorporen *información forward-looking*: las condiciones económicas actuales y las proyecciones futuras. Para eso, necesitamos construir dos piezas adicionales.


#### **PD PIT - ¿Dónde estamos hoy en el ciclo?**

La **PD PIT (Point-in-Time)** responde a la pregunta: *si el ambiente económico fuera permanentemente igual al de hoy, ¿cuál sería la tasa de default esperada?*

Se construye escalando la PD TTC por el ratio entre la tasa de default del período más reciente observado y el promedio histórico:

$$\text{mult}_{PIT} = \frac{\text{TD}_{actual}}{\overline{\text{TD}}_{histórica}}$$

$$\text{PD}_{PIT} = \text{PD}_{TTC} \times \text{mult}_{PIT}$$

> **Intuición:** Si la economía actual está por encima del promedio histórico (mayor desempleo, menor crecimiento), $\text{mult}_{PIT} > 1$ y la PD PIT será mayor que la PD TTC. Si estamos en un período de bonanza, $\text{mult}_{PIT} < 1$.


#### **PD Macro - ¿Hacia dónde vamos?**

La **PD macro** responde a la pregunta: *dadas las proyecciones económicas (base, adversa, optimista), ¿cuál es la tasa de default esperada?*

Se construye de forma análoga, pero usando la tasa de default proyectada —ponderada por la probabilidad de cada escenario— como referencia:

$$\text{TD}_{macro} = w_{base} \cdot \text{TD}_{base} + w_{adverso} \cdot \text{TD}_{adverso} + w_{optimista} \cdot \text{TD}_{optimista}$$

$$\text{mult}_{macro} = \frac{\text{TD}_{macro}}{\overline{\text{TD}}_{histórica}}$$

$$\text{PD}_{macro} = \text{PD}_{TTC} \times \text{mult}_{macro}$$

> **Intuición:** La PD macro captura el *futuro esperado*, no el presente. Si las proyecciones anticipan una recesión peor a las condiciones actuales, la PD macro superará a la PD PIT.


#### **PD Final IFRS 9 - Combinando los tres elementos**

Una vez que tenemos las tres PD, la PD final se obtiene directamente de la fórmula de la sección 2.3:

$$\text{PD}_{final} = \text{PD}_{TTC} \times \frac{\text{PD}_{macro}}{\text{PD}_{PIT}}$$

> **Intuición del ratio $\frac{\text{PD}_{macro}}{\text{PD}_{PIT}}$:** Este cociente es el *ajuste forward-looking relativo a las condiciones actuales*. Si las proyecciones son peores que hoy → ratio > 1 → la PD final sube respecto a la TTC. Si las proyecciones son mejores que hoy → ratio < 1 → la PD final baja. Nótese que si los escenarios proyectados coincidieran exactamente con las condiciones actuales, el ratio sería 1 y $\text{PD}_{final} = \text{PD}_{TTC}$.

Formalmente, el ratio se simplifica a:

$$\frac{\text{PD}_{macro}}{\text{PD}_{PIT}} = \frac{\text{PD}_{TTC} \times \text{mult}_{macro}}{\text{PD}_{TTC} \times \text{mult}_{PIT}} = \frac{\text{TD}_{macro}}{\text{TD}_{actual}}$$

Por lo tanto, la fórmula final equivale a:

$$\text{PD}_{final} = \text{PD}_{TTC} \times \frac{\text{TD}_{macro}}{\text{TD}_{actual}}$$

Es decir: partimos del ancla de largo plazo (TTC) y la ajustamos por cuánto esperamos que el futuro se aleje de las condiciones de hoy.

In [ ]:
# ============================================================
# PASO 3: PD PIT, PD MACRO Y PD FINAL (IFRS 9)
# ============================================================

# Tasas de default históricas observadas por año (simuladas)
# y sus correspondientes condiciones macroeconómicas
datos_macro = pd.DataFrame({
    'anio'           : [2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'tasa_default'   : [0.038, 0.041, 0.043, 0.078, 0.055, 0.048, 0.042],  # COVID spike en 2020
    'desempleo'      : [5.8,   5.6,   5.5,   9.2,   7.1,   6.2,   5.9],   # %
    'crecimiento_pib': [2.5,   2.8,   2.3,  -8.1,   3.8,   4.1,   2.6]    # % anual
})

# ── Referencia histórica de largo plazo ──────────────────────────────────────
# Es el "ancla" del modelo TTC: promedio del ciclo completo
tasa_default_historica = datos_macro['tasa_default'].mean()

# ── Condición actual (período de reporte) ────────────────────────────────────
# Usamos el año más reciente disponible como "estado actual del ciclo"
# En producción, esto correspondería al cierre del período de reporte
td_actual = datos_macro['tasa_default'].iloc[-1]  # 2023: 0.042

# ── Paso A: PD PIT — ajuste a condiciones actuales ───────────────────────────
# Intuición: qué tan distante está el ciclo actual del promedio histórico
mult_pit = td_actual / tasa_default_historica
df['pd_pit'] = (df['pd_ttc_calibrada'] * mult_pit).clip(0.001, 0.999)

print("PASO A — PD PIT (Condiciones Actuales)")
print("=" * 50)
print(f"  Tasa default histórica promedio : {tasa_default_historica:.4%}")
print(f"  Tasa default actual (2023)      : {td_actual:.4%}")
print(f"  Multiplicador PIT               : {mult_pit:.4f}  {'↓ por debajo del promedio' if mult_pit < 1 else '↑ por encima del promedio'}")
print(f"  PD TTC calibrada (promedio)     : {df['pd_ttc_calibrada'].mean():.4%}")
print(f"  PD PIT (promedio)               : {df['pd_pit'].mean():.4%}")

# ── Paso B: PD Macro — ajuste forward-looking con escenarios ─────────────────
# Proyecciones para el período siguiente al de reporte
escenario_base       = {'desempleo': 5.5, 'crecimiento_pib':  2.5}
escenario_adverso    = {'desempleo': 7.8, 'crecimiento_pib': -1.2}
escenario_optimista  = {'desempleo': 4.8, 'crecimiento_pib':  3.5}

# Regresión macro: tasa_default ~ desempleo (en la práctica se usan
# modelos más sofisticados con múltiples variables y rezagos)
X_macro = datos_macro[['desempleo']].values
y_macro = datos_macro['tasa_default'].values
reg_macro = LinearRegression().fit(X_macro, y_macro)

# Tasa de default proyectada por escenario
td_base      = reg_macro.predict([[escenario_base['desempleo']]])[0]
td_adverso   = reg_macro.predict([[escenario_adverso['desempleo']]])[0]
td_optimista = reg_macro.predict([[escenario_optimista['desempleo']]])[0]

# Ponderación estándar IFRS 9: los pesos deben reflejar la probabilidad
# asignada a cada escenario por el comité de riesgos de la entidad
peso_base, peso_adverso, peso_optimista = 0.50, 0.30, 0.20

# Tasa de default macro ponderada (numerador de la fórmula)
td_macro_ponderada = (peso_base * td_base +
                      peso_adverso * td_adverso +
                      peso_optimista * td_optimista)

mult_macro = td_macro_ponderada / tasa_default_historica
df['pd_macro'] = (df['pd_ttc_calibrada'] * mult_macro).clip(0.001, 0.999)

print()
print("PASO B — PD MACRO (Proyección Forward-Looking)")
print("=" * 50)
print(f"  Escenario base     (peso {peso_base:.0%}): TD proyectada = {td_base:.4%}  | Mult = {td_base/tasa_default_historica:.3f}")
print(f"  Escenario adverso  (peso {peso_adverso:.0%}): TD proyectada = {td_adverso:.4%}  | Mult = {td_adverso/tasa_default_historica:.3f}")
print(f"  Escenario optimist (peso {peso_optimista:.0%}): TD proyectada = {td_optimista:.4%}  | Mult = {td_optimista/tasa_default_historica:.3f}")
print(f"  TD macro ponderada              : {td_macro_ponderada:.4%}")
print(f"  Multiplicador macro             : {mult_macro:.4f}")
print(f"  PD macro (promedio)             : {df['pd_macro'].mean():.4%}")

# ── Paso C: PD FINAL IFRS 9 ──────────────────────────────────────────────────
# PD_final = PD_TTC × (PD_macro / PD_PIT)
#
# El ratio PD_macro / PD_PIT se simplifica a TD_macro / TD_actual,
# capturando cuánto se espera que cambie el ambiente crediticio
# respecto a las condiciones del período de reporte.
ajuste_forward_looking = df['pd_macro'] / df['pd_pit']  # = td_macro_ponderada / td_actual
df['pd_final'] = (df['pd_ttc_calibrada'] * ajuste_forward_looking).clip(0.001, 0.999)

ratio_macro_pit = td_macro_ponderada / td_actual  # factor de ajuste global

print()
print("PASO C — PD FINAL IFRS 9")
print("=" * 50)
print(f"  Fórmula: PD_final = PD_TTC × (PD_macro / PD_PIT)")
print(f"  Ratio forward-looking (TD_macro / TD_actual): {ratio_macro_pit:.4f}  "
      f"{'↑ macro peor que hoy' if ratio_macro_pit > 1 else '↓ macro mejor que hoy'}")
print()
print(f"  PD TTC calibrada (promedio)  : {df['pd_ttc_calibrada'].mean():.4%}  ← ancla histórica")
print(f"  PD PIT           (promedio)  : {df['pd_pit'].mean():.4%}  ← condiciones actuales")
print(f"  PD macro         (promedio)  : {df['pd_macro'].mean():.4%}  ← proyección ponderada")
print(f"  PD final IFRS 9  (promedio)  : {df['pd_final'].mean():.4%}  ← resultado final")
print("=" * 50)

In [ ]:
# ============================================================
# VISUALIZACIÓN: PD TTC, PD PIT, PD MACRO y PD FINAL
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Gráfico 1: Serie histórica + punto actual + proyecciones ---
ax = axes[0]
ax.plot(datos_macro['anio'], datos_macro['tasa_default'] * 100,
        'o-', color=PALETTE[0], linewidth=2.5, markersize=7, label='Tasa default histórica')

# Punto actual (2023) — referencia PD PIT
ax.scatter([2023], [td_actual * 100], color=PALETTE[0], s=120, zorder=6, edgecolors='black', linewidths=1.2)
ax.annotate(f'Actual\n{td_actual:.2%}', xy=(2023, td_actual * 100),
            xytext=(2022.2, td_actual * 100 + 0.6), fontsize=8.5, color=PALETTE[0])

# Proyecciones 2024 por escenario — referencia PD macro
anio_proy = 2024
for td_esc, label, color, idx in [
    (td_base,      f'Base ({td_base:.2%})',      PALETTE[0], 0),
    (td_adverso,   f'Adverso ({td_adverso:.2%})',  PALETTE[1], 1),
    (td_optimista, f'Optimista ({td_optimista:.2%})', PALETTE[2], 2),
]:
    ax.scatter([anio_proy], [td_esc * 100], color=color, s=80, zorder=5)
    ax.plot([2023, anio_proy], [td_actual * 100, td_esc * 100],
            '--', color=color, alpha=0.7, label=label)

# Punto PD macro ponderada
ax.scatter([anio_proy], [td_macro_ponderada * 100], color='black', s=100,
           zorder=7, marker='D', label=f'TD macro ponderada ({td_macro_ponderada:.2%})')

ax.axvline(2023.0, color='grey', linestyle=':', alpha=0.6)
ax.text(2022.1, ax.get_ylim()[1] * 0.97, 'Proyección →', fontsize=9, color='grey')
ax.set_xlabel('Año'); ax.set_ylabel('Tasa de Default (%)')
ax.set_title('Tasas de Default: Histórico, Actual y Proyecciones Forward-Looking')
ax.legend(fontsize=8.5); ax.grid(False)

# --- Gráfico 2: Distribución de las tres PD en la cartera ---
ax2 = axes[1]
ax2.hist(df['pd_ttc_calibrada'], bins=60, alpha=0.55, color=PALETTE[0], label='PD TTC calibrada', density=True)
ax2.hist(df['pd_pit'],           bins=60, alpha=0.55, color=PALETTE[2], label='PD PIT (condiciones actuales)', density=True)
ax2.hist(df['pd_macro'],         bins=60, alpha=0.55, color=PALETTE[1], label='PD macro (proyección forward)', density=True)
ax2.hist(df['pd_final'],         bins=60, alpha=0.70, color=PALETTE[3], label='PD final IFRS 9', density=True)

for col, color in [('pd_ttc_calibrada', PALETTE[0]), ('pd_pit', PALETTE[2]),
                   ('pd_macro', PALETTE[1]), ('pd_final', PALETTE[3])]:
    ax2.axvline(df[col].mean(), color=color, linestyle='--', linewidth=2)

ax2.set_xlabel('Probabilidad de Default'); ax2.set_ylabel('Densidad')
ax2.set_title('Distribución PD TTC / PIT / Macro / Final en la Cartera')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
ax2.legend(fontsize=8.5); ax2.grid(False)

plt.suptitle('PD Final IFRS 9: Construcción a Partir de TTC, PIT y Macro', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### **3.5 PD Lifetime: Curva de Supervivencia por Plazo**

Para los créditos en Stage 2 y Stage 3 (o cuando el plazo del crédito es mayor a 12 meses), IFRS 9 requiere calcular la **PD Lifetime**: la probabilidad de que el deudor incumpla en **cualquier momento** a lo largo del plazo residual del crédito.

La forma más sencilla de hacer esto es usar **matrices de probabilidad condicional de supervivencia**:

$$PD_{lifetime}(t) = 1 - \prod_{k=1}^{t} (1 - PD_{marginal,k})$$

Donde $PD_{marginal,k}$ es la probabilidad de caer en default en el período $k$, dado que se sobrevivió hasta $k-1$.

$$
PD_{marginal_t} = P(Default_t \mid {Vivo Hasta_{t-1}})
$$


In [ ]:
# ============================================================
# PASO 4: PD LIFETIME — CURVA DE SUPERVIVENCIA
# ============================================================

def calcular_pd_lifetime(pd_anual: float, horizonte_anios: float) -> float:
    """
    Calcula la PD Lifetime dado una PD anual (PIT) y un horizonte de tiempo.
    
    Usa el modelo de hazard constante: asume que la PD marginal anual
    es constante (simplificación), y descuenta la probabilidad de
    supervivencia acumulada.
    
    En la práctica, se usan matrices de transición que varían por año.
    """
    # Convertir la PD anual a una tasa de hazard mensual
    hazard_mensual = 1 - (1 - pd_anual) ** (1 / 12)
    n_meses = int(horizonte_anios * 12)
    
    # Probabilidad de supervivencia acumulada
    prob_supervivencia = (1 - hazard_mensual) ** n_meses
    
    return 1 - prob_supervivencia


def curva_pd_lifetime(pd_anual: float, max_anios: int = 5):
    """
    Genera la curva de PD Lifetime de 1 a max_anios años.
    """
    meses   = np.arange(1, max_anios * 12 + 1)
    pd_life = [calcular_pd_lifetime(pd_anual, m / 12) for m in meses]
    return meses, pd_life


# Calculamos la PD Lifetime para cada crédito de la cartera
df['horizonte_anios'] = df['plazo_meses'] / 12
df['pd_lifetime'] = df.apply(
    lambda r: calcular_pd_lifetime(r['pd_final'], r['horizonte_anios']), axis=1
).clip(0.001, 0.999)

# PD a 12 meses (para Stage 1)
df['pd_12m'] = df.apply(
    lambda r: calcular_pd_lifetime(r['pd_final'], 1.0), axis=1
).clip(0.001, 0.999)

print("PD RESULTANTES POR TIPO")
print("=" * 50)
print(f"  PD modelo original (TTC)    : {df['pd_modelo'].mean():.4%}")
print(f"  PD TTC calibrada            : {df['pd_ttc_calibrada'].mean():.4%}")
print(f"  PD final IFRS 9             : {df['pd_final'].mean():.4%}")
print(f"  PD 12 meses (Stage 1)       : {df['pd_12m'].mean():.4%}")
print(f"  PD Lifetime (Stage 2/3)     : {df['pd_lifetime'].mean():.4%}")
print("=" * 50)

In [ ]:
# ============================================================
# VISUALIZACIÓN: Curvas de PD Lifetime para distintos perfiles
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Gráfico 1: Curvas de PD Lifetime para distintas PD anuales ---
ax = axes[0]
max_anios = 5
perfiles_pd = [0.005, 0.02, 0.05, 0.10, 0.20]  # Perfiles de riesgo
etiquetas   = ['Muy bajo (0.5%)', 'Bajo (2%)', 'Medio (5%)', 'Alto (10%)', 'Muy alto (20%)']

for pd_anual, label, color in zip(perfiles_pd, etiquetas, PALETTE):
    meses, pd_life = curva_pd_lifetime(pd_anual, max_anios)
    ax.plot(meses / 12, [p * 100 for p in pd_life], label=label, color=color, linewidth=2.2)

ax.axvline(1, color='black', linestyle=':', alpha=0.5, label='12 meses (Stage 1)')
ax.set_xlabel('Horizonte (años)')
ax.set_ylabel('PD Lifetime (%)')
ax.set_title('Curvas de PD Lifetime por Perfil de Riesgo')
ax.legend(fontsize=9, title='Perfil de riesgo (PD anual)')
ax.grid(False)

# --- Gráfico 2: PD 12m vs PD Lifetime en la cartera ---
ax2 = axes[1]
ax2.hist(df['pd_12m'] * 100, bins=60, alpha=0.6, color=PALETTE[0], label='PD 12m (Stage 1)', density=True)
ax2.hist(df['pd_lifetime'] * 100, bins=60, alpha=0.6, color=PALETTE[4], label='PD Lifetime (Stage 2/3)', density=True)
ax2.set_xlabel('Probabilidad de Default (%)')
ax2.set_ylabel('Densidad')
ax2.set_title('Distribución PD 12m vs PD Lifetime en la Cartera')
ax2.legend()
ax2.grid(False)

plt.suptitle('PD Lifetime: Cuantificando el Riesgo a lo Largo del Plazo', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## **4. Clasificación en Stages y Cálculo del ECL**

El paso final es asignar cada crédito a su Stage correspondiente y calcular la **Pérdida Crediticia Esperada (ECL)**. Para simplificar, usaremos un LGD y EAD fijos por categoría de crédito. En la práctica, estos también se modelan.

In [ ]:
# ============================================================
# PASO 5: CLASIFICACIÓN EN STAGES Y CÁLCULO DE ECL
# ============================================================

# ── Definición de reglas de asignación a Stage ────────────────────
# Stage 1: PD final <= umbral SICR y no en default
# Stage 2: Incremento significativo en riesgo (SICR): PD final > 2x PD TTC (señal de deterioro)
# Stage 3: En default observado

# La PD TTC actúa como la PD de originación de referencia (sin ajuste de ciclo)
# El SICR compara la PD final (forward-looking) contra esa referencia estable
UMBRAL_SICR = 2.0  # Ratio PIT/TTC que activa el SICR

def asignar_stage(row):
    if row['default_obs'] == 1:
        return 3
    elif (row['pd_final'] / row['pd_ttc_calibrada']) >= UMBRAL_SICR:
        return 2
    else:
        return 1

df['stage'] = df.apply(asignar_stage, axis=1)

# ── Parámetros LGD y EAD ──────────────────────────────────────────
# En la práctica, LGD y EAD se modelan por producto/garantía/plazo.
# Aquí usamos supuestos simplificados.
LGD = 0.45          # Porcentaje perdido en caso de default (45%)
monto_credito = np.random.uniform(5_000, 80_000, N)  # EAD ≈ saldo adeudado
df['ead'] = monto_credito
df['lgd'] = LGD

# ── Cálculo del ECL según Stage ───────────────────────────────────
# Stage 1: ECL = PD 12m × LGD × EAD
# Stage 2: ECL = PD Lifetime × LGD × EAD
# Stage 3: ECL = LGD × EAD (el default ya ocurrió, PD = 1)

df['pd_ecl'] = np.where(
    df['stage'] == 1, df['pd_12m'],
    np.where(df['stage'] == 2, df['pd_lifetime'], 1.0)
)

df['ecl'] = df['pd_ecl'] * df['lgd'] * df['ead']

# ── Resumen de resultados ─────────────────────────────────────────
resumen_stages = df.groupby('stage').agg(
    n_creditos     = ('id_credito', 'count'),
    ead_total      = ('ead', 'sum'),
    pd_promedio    = ('pd_ecl', 'mean'),
    ecl_total      = ('ecl', 'sum')
).reset_index()
resumen_stages['cobertura_ecl'] = resumen_stages['ecl_total'] / resumen_stages['ead_total']
resumen_stages['pct_cartera']   = resumen_stages['n_creditos'] / N

print("RESUMEN ECL POR STAGE — REPORTE IFRS 9")
print("=" * 70)
for _, row in resumen_stages.iterrows():
    print(f"  Stage {int(row['stage'])}:")
    print(f"    Nº créditos   : {int(row['n_creditos']):>8,}  ({row['pct_cartera']:.1%} de la cartera)")
    print(f"    EAD total     : $ {row['ead_total']:>14,.0f}")
    print(f"    PD promedio   :   {row['pd_promedio']:>8.4%}")
    print(f"    ECL total     : $ {row['ecl_total']:>14,.0f}")
    print(f"    Cobertura ECL :   {row['cobertura_ecl']:>8.4%}")
    print()

ecl_total = df['ecl'].sum()
ead_total = df['ead'].sum()
print("=" * 70)
print(f"  ECL TOTAL CARTERA : $ {ecl_total:>14,.0f}")
print(f"  EAD TOTAL         : $ {ead_total:>14,.0f}")
print(f"  COBERTURA GLOBAL  :   {ecl_total / ead_total:.4%}")
print("=" * 70)

In [ ]:
# ============================================================
# VISUALIZACIÓN FINAL: Panel de resultados IFRS 9
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Colores por stage
stage_colors = {1: PALETTE[2], 2: PALETTE[3], 3: PALETTE[1]}
stage_labels = {1: 'Stage 1\n(Sin deterioro)', 2: 'Stage 2\n(Deteriorado)', 3: 'Stage 3\n(Default)'}

# --- Panel 1: Distribución de créditos por Stage ---
ax = axes[0, 0]
conteos = resumen_stages.set_index('stage')['n_creditos']
wedge_colors = [stage_colors[s] for s in conteos.index]
wedges, texts, autotexts = ax.pie(
    conteos, labels=[stage_labels[s] for s in conteos.index],
    autopct='%1.1f%%', colors=wedge_colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts: at.set_fontsize(10)
ax.set_title('Distribución de Cartera por Stage')

# --- Panel 2: ECL por Stage ---
ax2 = axes[0, 1]
ecl_por_stage = resumen_stages.set_index('stage')['ecl_total']
bars = ax2.bar([stage_labels[s] for s in ecl_por_stage.index],
               ecl_por_stage.values / 1e6,
               color=[stage_colors[s] for s in ecl_por_stage.index],
               edgecolor='white', linewidth=1.5)
ax2.set_ylabel('ECL (millones $)')
ax2.set_title('ECL Total por Stage')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f'${bar.get_height():.1f}M', ha='center', va='bottom', fontsize=10)

# --- Panel 3: Cobertura ECL/EAD por Stage ---
ax3 = axes[1, 0]
cob_por_stage = resumen_stages.set_index('stage')['cobertura_ecl']
bars3 = ax3.bar([stage_labels[s] for s in cob_por_stage.index],
                cob_por_stage.values * 100,
                color=[stage_colors[s] for s in cob_por_stage.index],
                edgecolor='white', linewidth=1.5)
ax3.set_ylabel('Cobertura ECL / EAD (%)')
ax3.set_title('Índice de Cobertura ECL por Stage')
for bar in bars3:
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             f'{bar.get_height():.2f}%', ha='center', va='bottom', fontsize=10)

# --- Panel 4: Evolución de PD en el proceso de calibración ---
ax4 = axes[1, 1]
etapas = ['PD Modelo\n(TTC cruda)', 'PD TTC\nCalibrada', 'PD PIT\n(Ajuste macro)', 'PD 12m\n(Stage 1)']
valores = [
    df['pd_modelo'].mean() * 100,
    df['pd_ttc_calibrada'].mean() * 100,
    df['pd_pit'].mean() * 100,
    df['pd_12m'].mean() * 100
]
bars4 = ax4.bar(etapas, valores, color=PALETTE[:4], edgecolor='white', linewidth=1.5)
ax4.set_ylabel('PD Promedio (%)')
ax4.set_title('Evolución de la PD a lo Largo del Proceso de Calibración')
for bar in bars4:
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}%', ha='center', va='bottom', fontsize=10)

plt.suptitle('Panel de Resultados IFRS 9 — Cartera de Consumo', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABLA RESUMEN FINAL — FORMATO REPORTE REGULATORIO IFRS 9
# ============================================================

reporte_final = resumen_stages.copy()
reporte_final['stage_label'] = reporte_final['stage'].map({
    1: 'Stage 1 — Sin deterioro significativo',
    2: 'Stage 2 — Incremento significativo de riesgo',
    3: 'Stage 3 — Activo deteriorado / Default'
})
reporte_final['pd_usada'] = reporte_final['stage'].map({
    1: 'PD 12m (PIT)',
    2: 'PD Lifetime (PIT)',
    3: 'PD = 100% (Default efectivo)'
})

print("REPORTE DE PROVISIONES BAJO IFRS 9")
print("Fecha de reporte: 31 de diciembre de 2023")
print("="*100)
print(f"{'Stage':<48} {'# Créditos':>11} {'EAD ($)':>15} {'PD usada':>22} {'ECL ($)':>14} {'Cobertura':>10}")
print("-"*100)
for _, row in reporte_final.iterrows():
    print(f"{row['stage_label']:<48} {int(row['n_creditos']):>11,} {row['ead_total']:>15,.0f} "
          f"{row['pd_usada']:>22} {row['ecl_total']:>14,.0f} {row['cobertura_ecl']:>9.3%}")

print("-"*100)
print(f"{'TOTAL CARTERA':<48} {N:>11,} {ead_total:>15,.0f} {'':>22} {ecl_total:>14,.0f} {ecl_total/ead_total:>9.3%}")
print("="*100)

## **6. Conclusiones**
En esta sesión aprendimos los conceptos fundamentales de IFRS 9 y aplicamos un proceso completo de calibración de PD. Los puntos más importantes son:

**Sobre IFRS 9:**
- IFRS 9 reemplazó el modelo de «pérdida incurrida» de IAS 39 por el modelo de **Pérdida Crediticia Esperada (ECL)**, mucho más prospectivo y preventivo.
- Las tres etapas (Stages) determinan si se usa una PD a 12 meses o una PD lifetime, siendo el SICR el detonante del paso de Stage 1 a Stage 2.

**Sobre la calibración de PD:**
- El modelo de scoring produce PD TTC, útil para rankings de riesgo pero no para provisionar directamente.
- La calibración TTC asegura que las PD del modelo sean consistentes con las tasas de default históricas reales.
- La **PD PIT** captura dónde estamos hoy en el ciclo económico: es el puente entre el largo plazo (TTC) y el presente.
- La **PD macro** captura hacia dónde vamos: incorpora los escenarios macroeconómicos ponderados exigidos por IFRS 9.
- La **PD final** combina los tres conceptos con la fórmula: $\text{PD}_{final} = \text{PD}_{TTC} \times \dfrac{\text{PD}_{macro}}{\text{PD}_{PIT}}$, donde el ratio actúa como ajuste forward-looking relativo a las condiciones actuales.
- La PD Lifetime se obtiene proyectando la PD final anual a través del horizonte de vida del crédito.

```
╔══════════════════════════════════════════════════════════════════════════╗
║         PROCESO COMPLETO DE CALIBRACIÓN PD BAJO IFRS 9                  ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  [1] MODELO DE SCORING  →  PD cruda del modelo (TTC, no calibrada)       ║
║             │                                                            ║
║             ▼                                                            ║
║  [2] CALIBRACIÓN TTC   →  Ajustar PD con tasas históricas observadas     ║
║             │              (Factor escala = Tasa real / PD modelo)       ║
║             ▼                                                            ║
║  [3a] PD PIT           →  PD TTC × (TD_actual / TD_histórica)           ║
║             │              ¿Dónde estamos hoy en el ciclo?               ║
║             │                                                            ║
║  [3b] PD MACRO         →  PD TTC × (TD_macro ponderada / TD_histórica)  ║
║             │              ¿Hacia dónde vamos según los escenarios?      ║
║             │                                                            ║
║  [3c] PD FINAL         →  PD TTC × (PD_macro / PD PIT)                  ║
║             │              = PD TTC × (TD_macro / TD_actual)             ║
║             ▼                                                            ║
║  [4] PD SEGÚN STAGE    →  Stage 1: PD final a 12 meses                  ║
║             │              Stage 2: PD Lifetime (curva de supervivencia) ║
║             │              Stage 3: PD = 100% (ya en default)            ║
║             ▼                                                            ║
║  [5] CÁLCULO ECL       →  ECL = PD × LGD × EAD                          ║
║             │                                                            ║
║             ▼                                                            ║
║  [6] REPORTE           →  Tabla de provisiones por Stage                 ║
║                            (reportada en estados financieros)            ║
╚══════════════════════════════════════════════════════════════════════════╝
```